# Paper Visualization Reproduction
This notebook reproduces Figure 1 (Main Results) and Figure 2 (Confusion Matrices) from the paper.

**Models evaluated:**
- Stage 1 Pretrain: One-class VAE baseline
- Method 1: Margin-based discriminative fine-tuning
- Method 2: Latent discriminator fine-tuning

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Set style for publication-quality figures
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Experimental Results Data
These metrics are from actual evaluation on 234 test samples (117 real + 117 fake).

In [ ]:
# Actual experimental results
results = {
    'Stage 1 Pretrain': {
        'Accuracy': 85.90,
        'Precision': 82.31,
        'Recall': 91.45,
        'Specificity': 80.34,
        'F1-Score': 86.64,
        'AUC-ROC': 85.90,
        'confusion_matrix': np.array([[94, 23], [10, 107]])
    },
    'Method 1: Margin': {
        'Accuracy': 88.46,
        'Precision': 85.71,
        'Recall': 92.31,
        'Specificity': 84.62,
        'F1-Score': 88.89,
        'AUC-ROC': 88.46,
        'confusion_matrix': np.array([[99, 18], [9, 108]])
    },
    'Method 2: Discriminator': {
        'Accuracy': 88.89,
        'Precision': 86.40,
        'Recall': 92.31,
        'Specificity': 85.47,
        'F1-Score': 89.26,
        'AUC-ROC': 88.89,
        'confusion_matrix': np.array([[100, 17], [9, 108]])
    }
}

metrics = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'AUC-ROC']
methods = list(results.keys())

print("Data loaded successfully!")
print(f"Methods: {methods}")
print(f"Metrics: {metrics}")

## Figure 1: Main Results Comparison
Bar charts showing performance across all metrics with summary table.

In [ ]:
# Create Figure 1: Main Results
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)

# Define colors for each method
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# Plot each metric in a subplot
for idx, metric in enumerate(metrics):
    row = idx // 3
    col = idx % 3
    ax = fig.add_subplot(gs[row, col])
    
    values = [results[method][metric] for method in methods]
    x = np.arange(len(methods))
    bars = ax.bar(x, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel(f'{metric} (%)', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Stage 1', 'Method 1', 'Method 2'], rotation=15, ha='right')
    ax.set_ylim([0, 100])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_title(metric, fontsize=12, fontweight='bold', pad=10)

# Add summary table in the remaining space
ax_table = fig.add_subplot(gs[2, :])
ax_table.axis('tight')
ax_table.axis('off')

# Prepare table data
table_data = []
for method in methods:
    row = [method]
    for metric in metrics:
        row.append(f"{results[method][metric]:.2f}")
    table_data.append(row)

# Create table
table = ax_table.table(cellText=table_data,
                       colLabels=['Method'] + metrics,
                       cellLoc='center',
                       loc='center',
                       bbox=[0, 0, 1, 1])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style the table
for i in range(len(methods)):
    table[(i+1, 0)].set_facecolor(colors[i])
    table[(i+1, 0)].set_alpha(0.3)

# Header row styling
for j in range(len(metrics) + 1):
    table[(0, j)].set_facecolor('#2C3E50')
    table[(0, j)].set_text_props(weight='bold', color='white')

plt.suptitle('Performance Comparison on Test Set (234 samples)', 
             fontsize=16, fontweight='bold', y=0.98)

plt.savefig('figure1_main_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure 1 saved as 'figure1_main_results.png'")

## Figure 2: Confusion Matrices
Confusion matrices for all three methods showing True Negatives (TN), False Positives (FP), False Negatives (FN), and True Positives (TP).

In [ ]:
# Create Figure 2: Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = [
    'Stage 1 Pretrain\nF1=86.64%, Acc=85.90%',
    'Method 1: Margin\nF1=88.89%, Acc=88.46%',
    'Method 2: Discriminator\nF1=89.26%, Acc=88.89%'
]

# Color maps for each method
cmaps = ['Reds', 'Blues', 'Greens']

for idx, (method, title, cmap) in enumerate(zip(methods, titles, cmaps)):
    cm = results[method]['confusion_matrix']
    ax = axes[idx]
    
    # Plot confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, 
                cbar=True, square=True, ax=ax,
                linewidths=2, linecolor='black',
                annot_kws={'size': 16, 'weight': 'bold'})
    
    # Add percentage annotations
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            percentage = (cm[i, j] / total) * 100
            ax.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)',
                   ha='center', va='center', fontsize=10, color='gray')
    
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
    ax.set_xticklabels(['Real', 'Fake'], fontsize=11)
    ax.set_yticklabels(['Real', 'Fake'], fontsize=11, rotation=0)

plt.suptitle('Confusion Matrices Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figure2_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure 2 saved as 'figure2_confusion_matrices.png'")

## Detailed Metrics Analysis

In [ ]:
# Print detailed analysis
print("="*70)
print("DETAILED PERFORMANCE ANALYSIS")
print("="*70)

for method in methods:
    print(f"\n{method}:")
    print("-" * 50)
    cm = results[method]['confusion_matrix']
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    
    print(f"  Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"  Accuracy:     {results[method]['Accuracy']:.2f}%")
    print(f"  Precision:    {results[method]['Precision']:.2f}%")
    print(f"  Recall:       {results[method]['Recall']:.2f}%")
    print(f"  Specificity:  {results[method]['Specificity']:.2f}%")
    print(f"  F1-Score:     {results[method]['F1-Score']:.2f}%")
    print(f"  AUC-ROC:      {results[method]['AUC-ROC']:.2f}%")

print("\n" + "="*70)
print("KEY FINDINGS:")
print("="*70)
print("✓ Method 2 (Discriminator) achieves best F1-score: 89.26%")
print("✓ Both Stage 2 methods significantly outperform Stage 1 baseline")
print("✓ Method 2 has highest specificity (85.47%) with 100 true negatives")
print("✓ Both Stage 2 methods achieve same recall (92.31%)")
print("="*70)

## Download Generated Figures
Run the following cell to download the generated figures to your local machine.

In [ ]:
# Download files (works in Google Colab)
try:
    from google.colab import files
    files.download('figure1_main_results.png')
    files.download('figure2_confusion_matrices.png')
    print("Files downloaded successfully!")
except ImportError:
    print("Not running in Colab. Files saved in current directory.")